In [0]:
%run "../../utils/00_config"

In [0]:
from pyspark.sql import functions as F

price_quality = spark.table(
    get_gold_table("intermediate", "int_item_price_quality")
)

price_history = spark.table(
    get_gold_table("marts", "mart_item_price_history")
)

dim_execution = spark.table(
    get_table("intermediate", "dim_execution")
)

print(f"Price quality records: {price_quality.count():,}")
print(f"Price history records: {price_history.count():,}")
print(f"Execution records: {dim_execution.count():,}")

In [0]:
# Aggregate offer quality by execution
quality_summary = (
    price_quality
    .groupBy("execution_id")
    .agg(
        F.count("*").alias("total_eligible_offers"),

        F.sum(
            F.when(F.col("price_quality_status") == "within_iqr", 1).otherwise(0)
        ).alias("within_iqr_offers"),

        F.sum(
            F.when(F.col("price_quality_status") == "outlier", 1).otherwise(0)
        ).alias("outlier_offers"),

        F.sum(
            F.when(F.col("price_quality_status") == "insufficient_sample", 1).otherwise(0)
        ).alias("insufficient_sample_offers")
    )
    .withColumn(
        "within_iqr_pct",
        F.round(F.col("within_iqr_offers") / F.col("total_eligible_offers") * 100, 2)
    )
    .withColumn(
        "outlier_pct",
        F.round(F.col("outlier_offers") / F.col("total_eligible_offers") * 100, 2)
    )
    .withColumn(
        "insufficient_sample_pct",
        F.round(F.col("insufficient_sample_offers") / F.col("total_eligible_offers") * 100, 2)
    )
)

display(quality_summary)

In [0]:
# Prepare execution metadata
execution_metadata = (
    dim_execution
    .select(
        "execution_id",
        F.col("started_at").alias("execution_started_at"),
        F.col("finished_at").alias("execution_finished_at"),
        "execution_duration_seconds",
        F.col("status").alias("execution_status"),
        F.col("response_count").alias("execution_response_count"),
        F.col("item_count").alias("execution_item_count"),
        F.col("total_orders").alias("execution_total_orders")
    )
)

# Validate execution uniqueness
duplicate_executions = (
    execution_metadata
    .groupBy("execution_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_executions > 0:
    raise ValueError("Duplicate execution_id found in execution metadata.")

# Combine quality and execution metrics
rel_market_quality = quality_summary.join(
    execution_metadata,
    on="execution_id",
    how="left"
)

display(rel_market_quality)

In [0]:
rel_market_quality = (
    rel_market_quality
    .withColumn(
        "eligible_offers_pct",
        F.round(
            F.col("total_eligible_offers")
            / F.col("execution_total_orders") * 100,
            2
        )
    )
    .withColumn(
        "excluded_offers",
        F.col("execution_total_orders") - F.col("total_eligible_offers")
    )
    .withColumn(
        "excluded_offers_pct",
        F.round(
            F.col("excluded_offers")
            / F.col("execution_total_orders") * 100,
            2
        )
    )
)

display(rel_market_quality)

In [0]:
validation = rel_market_quality.select(
    F.count("*").alias("total_executions"),
    F.sum("total_eligible_offers").alias("eligible_offers"),
    F.sum("within_iqr_offers").alias("within_iqr_offers"),
    F.sum("outlier_offers").alias("outlier_offers"),
    F.sum("insufficient_sample_offers").alias("insufficient_sample_offers"),
    F.sum("excluded_offers").alias("excluded_offers"),
    F.sum(
        F.when(
            F.col("total_eligible_offers") != (
                F.col("within_iqr_offers")
                + F.col("outlier_offers")
                + F.col("insufficient_sample_offers")
            ),
            1
        ).otherwise(0)
    ).alias("invalid_quality_breakdowns"),
    F.sum(
        F.when(
            F.col("execution_total_orders") != (
                F.col("total_eligible_offers") + F.col("excluded_offers")
            ),
            1
        ).otherwise(0)
    ).alias("invalid_eligibility_breakdowns")
)

display(validation)

In [0]:
# Define target table
TARGET_TABLE = get_gold_table(
    "reporting",
    "rel_market_quality"
)

# Materialize reporting table
(
    rel_market_quality
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Table successfully saved: {TARGET_TABLE}")

In [0]:
# Validate materialized table
saved_quality = spark.table(TARGET_TABLE)

print(f"Saved records: {saved_quality.count():,}")

display(saved_quality)